In [ ]:
# End-of-Course Project: RAG-Enhanced Financial Analyst 本项目旨在自动抽取A股上市公司配股预案中的“发行折价率”与“配股比例”，并计算提取准确率（Evals）。

In [4]:
import os
import json
import pandas as pd
from pydantic import BaseModel, Field
from openai import OpenAI

# 强烈建议：千万不要把真实的 API Key 写死在代码里传到 GitHub！
# 如果在本地运行，建议使用 dotenv；如果在 Colab 运行，使用 userdata
# 这里假设你已经在环境变量中配置好了
API_KEY = os.getenv("API_KEY", "？")
# 注意：去掉末尾的 /chat/completions
BASE_URL = "https://openrouter.ai/api/v1"

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

In [ ]:
## 1. 定义大模型结构化输出契约 (Structured JSON Contract) 为了避免大模型输出冗余文本，这里使用 Pydantic 定义严格的 JSON Schema。

In [5]:
class PlacementExtraction(BaseModel):
    stock_name: str = Field(description="上市公司的股票简称")
    placement_size: float = Field(description="配股比例，例如10配3，则输出0.3")
    discount_rate: float = Field(description="发行折价率，用小数表示")

def extract_financial_data(text_content: str, stock_name: str):
    prompt = f"""
    你是一个专业的金融分析师。请阅读以下A股上市公司的配股预案节选，
    直接提取出'配股比例'和'发行折价率'。

    注意：请在输出的 JSON 中，将 stock_name 字段严格固定为“{stock_name}”，绝对不要擅自使用公司全称。

    文本内容：{text_content}
    """

    response = client.beta.chat.completions.parse(
        model="openai/gpt-4o-mini",
        messages=[
            {"role": "system", "content": "严格输出JSON。绝对不要输出任何思维链、推理过程或解释文字。"},
            {"role": "user", "content": prompt}
        ],
        response_format=PlacementExtraction,
        max_tokens=300,
    )
    return response.choices[0].message.parsed

In [ ]:
## 2. 批量处理预案文本 (Batch Extraction) 读取 `data/announcements/` 目录下的文本文件，调用 API 进行信息抽取，并将结果持续追加写入 `evals/model_outputs.jsonl` 中。

In [6]:
import os

os.makedirs('evals', exist_ok=True)
output_jsonl_path = 'evals/model_outputs.jsonl'

print("=== 开始真实公告抽取测试 ===")

with open(output_jsonl_path, 'w', encoding='utf-8') as f_out:
    # 遍历你的标准答案名单
    for stock in ground_truth['股票简称']:
        # 指向你刚刚上传文件的真实路径
        file_path = f'announcements/{stock}.txt'

        if os.path.exists(file_path):
            with open(file_path, 'r', encoding='utf-8') as f_in:
                # 读取你上传的真实万字长文
                text_content = f_in.read()

            print(f"正在从真实长文本中提取: {stock}...")
            try:
                # 调用你设定好 Prompt 和 token 上限的大模型
                result = extract_financial_data(text_content, stock)
                f_out.write(result.model_dump_json() + '\n')
            except Exception as e:
                print(f"提取 {stock} 失败，原因: {e}")
        else:
            # 如果名字没对上或者漏传了，会在这里报警
            print(f"警告：找不到 {file_path}，请检查上传是否成功，跳过此样本。")

print("\n真实文本批量信息抽取完成！")

=== 开始真实公告抽取测试 ===
正在从真实长文本中提取: 浦发银行...
正在从真实长文本中提取: 兴业证券...
正在从真实长文本中提取: 贵州茅台...
正在从真实长文本中提取: 焦点科技...
正在从真实长文本中提取: 宁德时代...
正在从真实长文本中提取: 新天地...
正在从真实长文本中提取: 紫金矿业...
正在从真实长文本中提取: 东方证券...
正在从真实长文本中提取: 中马传动...
正在从真实长文本中提取: 佰维存储...
正在从真实长文本中提取: 西大门...
正在从真实长文本中提取: 爱美客...
正在从真实长文本中提取: 国药股份...
正在从真实长文本中提取: 思泰克...
正在从真实长文本中提取: 万科A...
正在从真实长文本中提取: 五粮液...
正在从真实长文本中提取: 中国平安...
正在从真实长文本中提取: 格力电器...
正在从真实长文本中提取: 伊利股份...
正在从真实长文本中提取: 美的集团...
正在从真实长文本中提取: 中国中免...
正在从真实长文本中提取: 隆基绿能...
正在从真实长文本中提取: 恒瑞医药...
正在从真实长文本中提取: 比亚迪...
正在从真实长文本中提取: 药明康德...

真实文本批量信息抽取完成！


In [ ]:
## 3. 自动化评估 (Evals) 将模型生成的 JSONL 结果与人工基准 CSV 进行对比，计算准确率指标。这一步取代了原先模糊的“缩短分析时间”指标。

In [7]:
def run_evals():
    truth_dict = ground_truth.set_index('股票简称').to_dict('index')

    total_cases = 0
    size_correct = 0
    discount_correct = 0

    with open(output_jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            model_out = json.loads(line)
            stock = model_out['stock_name']

            if stock in truth_dict:
                total_cases += 1
                truth_row = truth_dict[stock]

                # 容差范围设为 0.01 以应对浮点数精度问题
                if abs(model_out['placement_size'] - truth_row['配股比例']) < 0.01:
                    size_correct += 1
                if abs(model_out['discount_rate'] - truth_row['发行折价率']) < 0.01:
                    discount_correct += 1

    print(f"=== Evals 评估报告 ===")
    if total_cases == 0:
        print("暂无测试结果，请确保 API 抽取步骤已成功运行。")
        return

    print(f"总测试样本数: {total_cases}")
    print(f"配股比例 (Placement Size) 提取准确率: {size_correct}/{total_cases} ({size_correct/total_cases*100:.1f}%)")
    print(f"发行折价率 (Discount Rate) 提取准确率: {discount_correct}/{total_cases} ({discount_correct/total_cases*100:.1f}%)")

run_evals()

=== Evals 评估报告 ===
总测试样本数: 25
配股比例 (Placement Size) 提取准确率: 25/25 (100.0%)
发行折价率 (Discount Rate) 提取准确率: 25/25 (100.0%)
